# Time Series Course — m10-f2 (block 2)

_From the **Time Series & Forecasting** course on Zylo._

Run all cells (`Runtime → Run all` or `Ctrl+F9`) to execute.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Assume: train_series, test_series are 1D numpy arrays
# Step 1: Normalize with TRAINING stats only
train_mean = train_series.mean()
train_std = train_series.std()
train_norm = (train_series - train_mean) / train_std
test_norm = (test_series - train_mean) / train_std  # same stats!

# Step 2: Create windows
lookback = 60
X_train, y_train = create_windows(train_norm, lookback)
X_test, y_test = create_windows(test_norm, lookback)

# Step 3: Convert to tensors — add feature dimension
X_train_t = torch.FloatTensor(X_train).unsqueeze(-1)  # (N, 60, 1)
y_train_t = torch.FloatTensor(y_train).unsqueeze(-1)  # (N, 1)
X_test_t = torch.FloatTensor(X_test).unsqueeze(-1)
y_test_t = torch.FloatTensor(y_test).unsqueeze(-1)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=32, shuffle=True  # shuffle batches (not within batches)
)

# Step 4: Train
model = LSTMForecaster(input_size=1, hidden_size=64, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(50):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test_t)
            test_loss = criterion(test_pred, y_test_t).item()
        print(f"Epoch {epoch+1}: train_loss={epoch_loss/len(train_loader):.4f}, test_loss={test_loss:.4f}")